# Subtask 3 — Document-Grounded Dialogue Generation in Hinglish

The goal of this experiment is to extend the attention-based sequence-to-sequence architecture from Subtask 2 to multi-turn, document-grounded dialogue generation.

The model receives two conditioning sources:

1. Conversation history
2. A grounding document

A custom tokenizer and trainable embedding layer are used to support Hinglish/code-mixed text without relying on pretrained language models.

Two models are compared:

- **Ungrounded baseline:** uses conversation history only.
- **Grounded model:** uses both conversation history and the grounding document.

The models generate Hinglish responses token-by-token.

In [1]:
# Load the CMU Hinglish DoG dataset
!pip -q install datasets rouge_score

from datasets import load_dataset

dataset = load_dataset("festvox/cmu_hinglish_dog")

print(dataset)

  Preparing metadata (setup.py) ... done


README.md:   0%|          | 0.00/8.30k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  844kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 99.6kB            

...

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8060 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/960 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/942 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['date', 'docIdx', 'translation', 'uid', 'utcTimestamp', 'rating', 'status', 'uid1LogInTime', 'uid1LogOutTime', 'uid1response', 'uid2response', 'user2_id', 'whoSawDoc', 'wikiDocumentIdx'],
        num_rows: 8060
    })
    test: Dataset({
        features: ['date', 'docIdx', 'translation', 'uid', 'utcTimestamp', 'rating', 'status', 'uid1LogInTime', 'uid1LogOutTime', 'uid1response', 'uid2response', 'user2_id', 'whoSawDoc', 'wikiDocumentIdx'],
        num_rows: 960
    })
    validation: Dataset({
        features: ['date', 'docIdx', 'translation', 'uid', 'utcTimestamp', 'rating', 'status', 'uid1LogInTime', 'uid1LogOutTime', 'uid1response', 'uid2response', 'user2_id', 'whoSawDoc', 'wikiDocumentIdx'],
        num_rows: 942
    })
})


In [2]:
# Inspect the dataset structure and one example
print(dataset.column_names)

for split in dataset:
    print("\nSPLIT:", split)
    print("Rows:", len(dataset[split]))
    print("Columns:", dataset[split].column_names)
    print("Example:", dataset[split][0])

{'train': ['date', 'docIdx', 'translation', 'uid', 'utcTimestamp', 'rating', 'status', 'uid1LogInTime', 'uid1LogOutTime', 'uid1response', 'uid2response', 'user2_id', 'whoSawDoc', 'wikiDocumentIdx'], 'test': ['date', 'docIdx', 'translation', 'uid', 'utcTimestamp', 'rating', 'status', 'uid1LogInTime', 'uid1LogOutTime', 'uid1response', 'uid2response', 'user2_id', 'whoSawDoc', 'wikiDocumentIdx'], 'validation': ['date', 'docIdx', 'translation', 'uid', 'utcTimestamp', 'rating', 'status', 'uid1LogInTime', 'uid1LogOutTime', 'uid1response', 'uid2response', 'user2_id', 'whoSawDoc', 'wikiDocumentIdx']}

SPLIT: train
Rows: 8060
Columns: ['date', 'docIdx', 'translation', 'uid', 'utcTimestamp', 'rating', 'status', 'uid1LogInTime', 'uid1LogOutTime', 'uid1response', 'uid2response', 'user2_id', 'whoSawDoc', 'wikiDocumentIdx']
Example: {'date': '2018-03-16T20:49:27.955Z', 'docIdx': 0, 'translation': {'en': 'hi', 'hi_en': 'hi'}, 'uid': 'user1', 'utcTimestamp': '2018-03-16T20:49:39.626Z', 'rating': 2, 'st

## Dataset Structure and Preparation

The CMU Hinglish DoG dataset contains 8,060 training rows, 942 validation rows, and 960 test rows.

Each row contains a Hinglish/English dialogue utterance in the `translation` field, speaker information, timestamps, conversation/document identifiers, and `wikiDocumentIdx`.

The released splits expose the `wikiDocumentIdx` identifier but do not directly provide the corresponding Wikipedia document text. Therefore, for this implementation, the grounding context is constructed from the available utterances associated with the same Wikipedia document index. This provides a separate document-level context while keeping the experiment reproducible using only the supplied dataset.

The target response is the next Hinglish utterance, while the input consists of previous dialogue history and the document-level grounding context.

In [3]:
# Build dialogue-history, grounding-context, and response examples
def get_hinglish(row):
    value = row["translation"]
    if isinstance(value, dict):
        return str(value.get("hi_en", value.get("en", "")))
    return str(value)

def build_examples(split_data, max_rows=None):
    rows = list(split_data)
    rows.sort(key=lambda x: x["utcTimestamp"])

    # Group available utterances by wiki document index
    document_groups = {}

    for row in rows:
        doc_id = row["wikiDocumentIdx"]
        text = get_hinglish(row).strip()

        if text:
            document_groups.setdefault(doc_id, []).append(text)

    examples = []

    # Create examples using preceding utterances as dialogue history
    for i in range(1, len(rows)):
        current = rows[i]
        current_text = get_hinglish(current).strip()

        if not current_text:
            continue

        # Use nearby previous turns as conversation history
        history_parts = []

        for j in range(max(0, i - 4), i):
            previous_text = get_hinglish(rows[j]).strip()
            if previous_text:
                history_parts.append(previous_text)

        if not history_parts:
            continue

        history = " ".join(history_parts)

        # Grounding context from the same wiki document index
        doc_id = current["wikiDocumentIdx"]
        document = " ".join(document_groups.get(doc_id, []))

        if not document:
            continue

        examples.append({
            "history": history,
            "document": document,
            "response": current_text
        })

        if max_rows and len(examples) >= max_rows:
            break

    return examples

train_examples = build_examples(dataset["train"], max_rows=5000)
val_examples = build_examples(dataset["validation"], max_rows=500)
test_examples = build_examples(dataset["test"], max_rows=500)

print("Train:", len(train_examples))
print("Validation:", len(val_examples))
print("Test:", len(test_examples))

print("\nExample:")
print("History:", train_examples[0]["history"])
print("Document:", train_examples[0]["document"][:500])
print("Response:", train_examples[0]["response"])

Train: 5000
Validation: 500
Test: 500

Example:
History: hello, Namaste.
Document: hello, Namaste. Namaskar, kya tumne John Wick movies watch ki? mene first one dekhi but mene second one nahi dekhi. Next one great he, jahan voh apni stolen car ko search karne jata he. ye to interesting lagta he. me 2nd John Wick movie dekhne ka plan banaunga.Movie ka topic kya tha jiske bare me tumne read kiya tha. voh toh apni car ka chori ko revenge karte karte usne puppy ko maar daali konsa person ko voh revenge lena chatha hein Losef us Tarasov ka head tha, usne Mustant ko Aurello ka shop 
Response: Namaskar, kya tumne John Wick movies watch ki?


In [4]:
# Create a Unicode-aware tokenizer and vocabulary for Hinglish
import re
from collections import Counter

def tokenize_hinglish(text):
    text = str(text).lower()
    return re.findall(
        r"[^\W_]+(?:['’][^\W_]+)?|[^\w\s]",
        text,
        flags=re.UNICODE
    )

SPECIAL_TOKENS = {
    "<PAD>": 0,
    "<SOS>": 1,
    "<EOS>": 2,
    "<UNK>": 3
}

def build_vocab(examples, max_vocab=15000, min_freq=2):
    counter = Counter()

    for example in examples:
        for field in ["history", "document", "response"]:
            counter.update(tokenize_hinglish(example[field]))

    vocab = dict(SPECIAL_TOKENS)

    for token, freq in counter.most_common():
        if freq < min_freq:
            break
        if token not in vocab:
            vocab[token] = len(vocab)

        if len(vocab) >= max_vocab:
            break

    return vocab

vocab = build_vocab(train_examples)

print("Vocabulary size:", len(vocab))
print(list(vocab.items())[:20])

Vocabulary size: 7408
[('<PAD>', 0), ('<SOS>', 1), ('<EOS>', 2), ('<UNK>', 3), ('hai', 4), ('.', 5), (',', 6), ('ke', 7), ('ki', 8), ('?', 9), ('mein', 10), ('ko', 11), ('aur', 12), ('kya', 13), ('mujhe', 14), ('movie', 15), ('tha', 16), ('ek', 17), ('me', 18), ('nahi', 19)]


In [5]:
# Encode dialogue fields and limit sequence lengths for faster training
PAD_ID = vocab["<PAD>"]
SOS_ID = vocab["<SOS>"]
EOS_ID = vocab["<EOS>"]
UNK_ID = vocab["<UNK>"]

MAX_HISTORY = 60
MAX_DOCUMENT = 100
MAX_RESPONSE = 40

def encode_text(text, max_length):
    tokens = tokenize_hinglish(text)[:max_length - 2]

    ids = [SOS_ID]

    for token in tokens:
        ids.append(vocab.get(token, UNK_ID))

    ids.append(EOS_ID)

    return ids

def encode_examples(examples):
    histories = []
    documents = []
    responses = []

    for example in examples:
        histories.append(
            encode_text(example["history"], MAX_HISTORY)
        )
        documents.append(
            encode_text(example["document"], MAX_DOCUMENT)
        )
        responses.append(
            encode_text(example["response"], MAX_RESPONSE)
        )

    return histories, documents, responses

train_history, train_document, train_response = encode_examples(train_examples)
val_history, val_document, val_response = encode_examples(val_examples)
test_history, test_document, test_response = encode_examples(test_examples)

print(len(train_history), len(train_document), len(train_response))

5000 5000 5000


In [6]:
# Create padded TensorFlow datasets for dialogue generation
import tensorflow as tf

BATCH_SIZE = 32

def make_dataset(histories, documents, responses, shuffle=False):
    ds = tf.data.Dataset.from_generator(
        lambda: zip(histories, documents, responses),
        output_signature=(
            tf.TensorSpec(shape=(None,), dtype=tf.int32),
            tf.TensorSpec(shape=(None,), dtype=tf.int32),
            tf.TensorSpec(shape=(None,), dtype=tf.int32)
        )
    )

    if shuffle:
        ds = ds.shuffle(5000)

    ds = ds.padded_batch(
        BATCH_SIZE,
        padded_shapes=([None], [None], [None]),
        padding_values=(PAD_ID, PAD_ID, PAD_ID)
    )

    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(
    train_history, train_document, train_response, shuffle=True
)

val_ds = make_dataset(
    val_history, val_document, val_response
)

test_ds = make_dataset(
    test_history, test_document, test_response
)

print("Datasets ready.")

Datasets ready.


In [7]:
# Build a two-source attention-based Hinglish dialogue model
EMBEDDING_DIM = 64
UNITS = 128

class DialogueAttentionModel(tf.keras.Model):
    def __init__(self, vocab_size):
        super().__init__()

        self.embedding = tf.keras.layers.Embedding(
            vocab_size,
            EMBEDDING_DIM
        )

        self.history_encoder = tf.keras.layers.LSTM(
            UNITS,
            return_sequences=True,
            return_state=True
        )

        self.document_encoder = tf.keras.layers.LSTM(
            UNITS,
            return_sequences=True,
            return_state=True
        )

        self.decoder = tf.keras.layers.LSTM(
            UNITS,
            return_sequences=True,
            return_state=True
        )

        self.W1 = tf.keras.layers.Dense(UNITS)
        self.W2 = tf.keras.layers.Dense(UNITS)
        self.V = tf.keras.layers.Dense(1)

        self.output_layer = tf.keras.layers.Dense(vocab_size)

    def attention(self, query, values):
        query = tf.expand_dims(query, 1)

        score = self.V(
            tf.nn.tanh(
                self.W1(values) + self.W2(query)
            )
        )

        weights = tf.nn.softmax(score, axis=1)

        context = tf.reduce_sum(
            weights * values,
            axis=1
        )

        return context

    def call(self, history, document, decoder_input):
        history_emb = self.embedding(history)
        document_emb = self.embedding(document)
        decoder_emb = self.embedding(decoder_input)

        history_out, hh, hc = self.history_encoder(history_emb)
        document_out, dh, dc = self.document_encoder(document_emb)

        # Combine information from both sources
        combined = tf.concat(
            [history_out, document_out],
            axis=1
        )

        decoder_out, _, _ = self.decoder(
            decoder_emb,
            initial_state=[hh, hc]
        )

        # Attention over both history and document
        contexts = []

        target_length = tf.shape(decoder_out)[1]
        i = tf.constant(0)

        context_array = tf.TensorArray(
            dtype=decoder_out.dtype,
            size=target_length
        )

        def condition(i, context_array):
            return i < target_length

        def body(i, context_array):
            context = self.attention(
                decoder_out[:, i, :],
                combined
            )
            context_array = context_array.write(i, context)
            return i + 1, context_array

        _, context_array = tf.while_loop(
            condition,
            body,
            [i, context_array]
        )

        contexts = tf.transpose(
            context_array.stack(),
            [1, 0, 2]
        )

        merged = tf.concat(
            [decoder_out, contexts],
            axis=-1
        )

        return self.output_layer(merged)

grounded_model = DialogueAttentionModel(len(vocab))

optimizer = tf.keras.optimizers.Adam(0.001)

print("Grounded model created.")

Grounded model created.


In [9]:
# Create the masked loss and training step
loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)

def dialogue_loss(real, predictions):
    loss = loss_object(real, predictions)
    mask = tf.cast(real != PAD_ID, loss.dtype)
    loss *= mask

    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

@tf.function(reduce_retracing=True)
def grounded_train_step(history, document, response):
    decoder_input = response[:, :-1]
    target = response[:, 1:]

    with tf.GradientTape() as tape:
        predictions = grounded_model(
            history,
            document,
            decoder_input
        )

        loss = dialogue_loss(
            target,
            predictions
        )

    gradients = tape.gradient(
        loss,
        grounded_model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, grounded_model.trainable_variables)
    )

    return loss

In [10]:
# Train the grounded dialogue model for two lightweight epochs
import time

for epoch in range(2):
    start = time.time()
    total_loss = 0.0
    batches = 0

    for history, document, response in train_ds:
        loss = grounded_train_step(
            history,
            document,
            response
        )

        total_loss += float(loss)
        batches += 1

    print(
        f"Epoch {epoch+1} | "
        f"Loss: {total_loss/batches:.4f} | "
        f"Time: {(time.time()-start)/60:.2f} min"
    )

Epoch 1 | Loss: 6.6870 | Time: 3.80 min
Epoch 2 | Loss: 6.0697 | Time: 4.37 min


In [11]:
# Generate sample grounded Hinglish responses
for i in range(min(5, len(test_examples))):
    example = test_examples[i]

    generated = generate_response(
        example["history"],
        example["document"]
    )

    print("\n" + "=" * 80)
    print("HISTORY:", example["history"])
    print("DOCUMENT:", example["document"][:300])
    print("REFERENCE:", example["response"])
    print("GENERATED:", generated)

NameError: name 'generate_response' is not defined

In [12]:
# Define the response generator for the trained grounded model
id_to_token = {v: k for k, v in vocab.items()}

def generate_response(history_text, document_text, max_length=30):
    history = tf.constant([
        encode_text(history_text, MAX_HISTORY)
    ])

    document = tf.constant([
        encode_text(document_text, MAX_DOCUMENT)
    ])

    generated = [SOS_ID]

    for _ in range(max_length):
        decoder_input = tf.constant([generated])

        predictions = grounded_model(
            history,
            document,
            decoder_input
        )

        next_token = int(
            tf.argmax(predictions[0, -1]).numpy()
        )

        if next_token == EOS_ID:
            break

        generated.append(next_token)

    tokens = [
        id_to_token.get(i, "<UNK>")
        for i in generated[1:]
    ]

    return " ".join(tokens)

In [13]:
# Generate sample grounded Hinglish responses
for i in range(min(5, len(test_examples))):
    example = test_examples[i]

    generated = generate_response(
        example["history"],
        example["document"]
    )

    print("\n" + "=" * 80)
    print("HISTORY:", example["history"])
    print("DOCUMENT:", example["document"][:300])
    print("REFERENCE:", example["response"])
    print("GENERATED:", generated)


HISTORY: hi
DOCUMENT: hi hi Marvel's The Avenger 2012 ki Amaerica superhero film hai jo isi naam ki Marvel Comics Superhero team par based hai, Marvel studios ne isey produce kiya ha aur Wallt DisneyStudios Motion pictures ne isey distribute kiya hai Marvel's The Avengers 2012 kee American superhero philm hai, jo isee na
REFERENCE: hi
GENERATED: hai

HISTORY: hi hi
DOCUMENT: hi hi Marvel's The Avenger 2012 ki Amaerica superhero film hai jo isi naam ki Marvel Comics Superhero team par based hai, Marvel studios ne isey produce kiya ha aur Wallt DisneyStudios Motion pictures ne isey distribute kiya hai Marvel's The Avengers 2012 kee American superhero philm hai, jo isee na
REFERENCE: Marvel's The Avenger 2012 ki Amaerica superhero film hai jo isi naam ki Marvel Comics Superhero team par based hai, Marvel studios ne isey produce kiya ha aur Wallt DisneyStudios Motion pictures ne isey distribute kiya hai
GENERATED: hai

HISTORY: hi hi Marvel's The Avenger 2012 ki Amaerica superhero film ha

**1. Run BLEU + ROUGE now**

In [14]:
# Evaluate the grounded model on 100 test examples
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

rouge = rouge_scorer.RougeScorer(
    ["rouge1", "rougeL"],
    use_stemmer=True
)

smooth = SmoothingFunction().method1

bleu_scores = []
rouge1_scores = []
rougeL_scores = []

for example in test_examples[:100]:

    generated = generate_response(
        example["history"],
        example["document"]
    )

    reference = example["response"]

    generated_tokens = tokenize_hinglish(generated)
    reference_tokens = tokenize_hinglish(reference)

    if not generated_tokens:
        continue

    bleu = sentence_bleu(
        [reference_tokens],
        generated_tokens,
        smoothing_function=smooth
    )

    scores = rouge.score(
        reference,
        generated
    )

    bleu_scores.append(bleu)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print(f"Grounded BLEU: {sum(bleu_scores)/len(bleu_scores):.6f}")
print(f"Grounded ROUGE-1: {sum(rouge1_scores)/len(rouge1_scores):.6f}")
print(f"Grounded ROUGE-L: {sum(rougeL_scores)/len(rougeL_scores):.6f}")

Grounded BLEU: 0.015178
Grounded ROUGE-1: 0.094373
Grounded ROUGE-L: 0.094373


In [16]:
# Define the ungrounded inference setting
def generate_ungrounded(history_text, max_length=30):
    return generate_response(
        history_text,
        "",
        max_length
    )

In [17]:
# Compare grounded and ungrounded generation on five test examples
for i in range(min(5, len(test_examples))):
    example = test_examples[i]

    grounded = generate_response(
        example["history"],
        example["document"]
    )

    ungrounded = generate_ungrounded(
        example["history"]
    )

    print("\n" + "=" * 80)
    print("Example", i + 1)
    print("HISTORY:", example["history"][-500:])
    print("REFERENCE:", example["response"])
    print("UNGROUNDED:", ungrounded)
    print("GROUNDED:", grounded)


Example 1
HISTORY: hi
REFERENCE: hi
UNGROUNDED: 
GROUNDED: hai

Example 2
HISTORY: hi hi
REFERENCE: Marvel's The Avenger 2012 ki Amaerica superhero film hai jo isi naam ki Marvel Comics Superhero team par based hai, Marvel studios ne isey produce kiya ha aur Wallt DisneyStudios Motion pictures ne isey distribute kiya hai
UNGROUNDED: 
GROUNDED: hai

Example 3
HISTORY: hi hi Marvel's The Avenger 2012 ki Amaerica superhero film hai jo isi naam ki Marvel Comics Superhero team par based hai, Marvel studios ne isey produce kiya ha aur Wallt DisneyStudios Motion pictures ne isey distribute kiya hai
REFERENCE: Marvel's The Avengers 2012 kee American superhero philm hai, jo isee naam kee maarval komiks suparaheero teem par aadhaarit hai, jo maarval stoodiyoj dvaara nirmit aur volt dizaneestoodios moshan pikchars dvaara vitarit kee gaee hai.
UNGROUNDED: kya , , ,
GROUNDED: haan , , hai

Example 4
HISTORY: hi hi Marvel's The Avenger 2012 ki Amaerica superhero film hai jo isi naam ki Marvel Comic

# Subtask 3 — Final Results and Takeaways

## Dataset and Approach

The CMU Hinglish DoG dataset was used for document-grounded dialogue generation. The released data contains Hinglish/English dialogue turns together with conversation metadata and `wikiDocumentIdx`.

A custom Unicode-aware tokenizer was created to handle code-mixed Hinglish text. The vocabulary and embedding layer were learned from scratch without using pretrained language models.

The grounded model contains two LSTM encoders:

- **History encoder:** processes the previous conversation turns.
- **Document encoder:** processes the grounding context associated with the same document index.

The decoder uses Bahdanau-style attention over the combined encoder representations and generates the response token-by-token.

An ungrounded inference setting was also evaluated by removing the document context.

## Training

The grounded model was trained on a reduced subset because of computational constraints.

| Epoch | Loss | Time |
|---|---:|---:|
| 1 | 6.6870 | 3.80 min |
| 2 | 6.0697 | 4.37 min |

The decreasing training loss indicates that the model was learning the dialogue generation task.

## Automatic Evaluation

Evaluation was performed on 100 held-out test examples.

| Metric | Grounded Model |
|---|---:|
| BLEU | **0.015178** |
| ROUGE-1 | **0.094373** |
| ROUGE-L | **0.094373** |

The relatively low scores indicate that the generated responses often differed substantially from the reference responses.

## Qualitative Analysis

The model frequently generated short generic responses such as:

- `hai`
- `haan, hai`
- `kya, hai`

The ungrounded setting also frequently produced degenerate or empty responses. The grounded model generally produced a non-empty response, but it did not consistently generate detailed responses containing the information present in the reference.

This shows that although the model successfully implements the required history + document conditioning pipeline, substantially more training data and training time would be required for strong dialogue quality.

## Grounding and Limitations

The released dataset splits expose `wikiDocumentIdx`, but the corresponding original Wikipedia document text is not directly included in the dataset rows used here. Therefore, the grounding context was constructed from the available utterances associated with the same document index.

The experiment was also constrained to 5,000 training examples and two epochs due to computational/time limitations.

## Key Takeaways

1. The task was successfully extended from single-sentence translation to multi-turn Hinglish dialogue generation.
2. A custom tokenizer and trainable embeddings were used instead of pretrained language models.
3. The model uses separate encoders for conversation history and grounding context.
4. Bahdanau attention allows the decoder to attend over information from both sources.
5. Automatic metrics were low, reflecting the limited training budget and the difficulty of generating code-mixed conversational responses.
6. Qualitative results show that the grounded model can produce responses, but further training would be necessary to obtain richer and more contextually appropriate dialogue.

## Conclusion

Subtask 3 demonstrates an end-to-end document-conditioned Hinglish dialogue generation pipeline using encoder-decoder architecture with attention. Compared with the translation experiments, this task is substantially more challenging because dialogue has multiple valid responses and requires maintaining conversational context while using external grounding information.